# CMS-HCC Batch Risk Score Processing

This notebook demonstrates how to calculate Hierarchical Condition Category (HCC) risk adjustment scores on large datasets (such as CSV or Excel files containing thousands of patient records) using the high-performance `HCCBatchProcessor` module.

In [ ]:
# Install required libraries
!pip install hccinfhir pandas openpyxl matplotlib tqdm

## Step 1: Basic Usage (Single Patient)

Verify that the `hccinfhir` library is working correctly by running a single-patient calculation.

In [ ]:
from hccinfhir import HCCInFHIR

processor = HCCInFHIR(model_name="CMS-HCC Model V28")

result = processor.calculate_from_diagnosis(["E11.9", "I10", "N18.3"], age=67, sex="F")
print(f"Risk Score: {result.risk_score}")
print(f"HCCs: {result.hcc_list}")
print(f"HCC Details: {result.hcc_details}")

## Step 2: Generate Mock Patient Data for Scalability Testing

To test the performance of the batch processor on "huge data," we can generate a mock dataset containing thousands of patients with realistic age, gender, and ICD-10 diagnosis codes distributions.

In [ ]:
from hcc_batch_processor import HCCBatchProcessor

# Generate 50,000 mock patients and save them to a CSV
mock_file = "mock_patients_large.csv"
mock_df = HCCBatchProcessor.generate_mock_data(num_records=50000, output_path=mock_file)

# Preview the generated data
print("\nData Preview:")
mock_df.head(10)

## Step 3: Run Batch Calculations on Large Data

Now, let's use the `HCCBatchProcessor` to score the 50,000 mock patients. We'll use `chunksize` to process in chunks to prevent memory overhead, writing the results incrementally to the output CSV.

In [ ]:
# Initialize the batch processor
batch_proc = HCCBatchProcessor(model_name="CMS-HCC Model V28")

# Define output path
output_file = "mock_patients_large_scored.csv"

# Process the file
stats = batch_proc.process_file(
    input_path=mock_file,
    output_path=output_file,
    chunksize=10000  # Process in chunks of 10,000 rows to keep memory usage low
)

## Step 4: Load and Inspect Scored Output

Let's read the output file to examine the new fields appended by the batch processor: `risk_score`, `risk_score_demographics`, `risk_score_hcc`, `hcc_list`, `hcc_details_summary`, and `error_log`.

In [ ]:
import pandas as pd

scored_df = pd.read_csv(output_file)
print(f"Loaded scored dataset shape: {scored_df.shape}")

# Filter out rows with errors for a cleaner look
successful_rows = scored_df[scored_df["error_log"].isna() | (scored_df["error_log"] == "")]
print(f"Successful records: {len(successful_rows)} / {len(scored_df)}")

print("\nSample Successful Scores:")
successful_rows[["PatientID", "Age", "Gender", "Diagnoses", "risk_score", "hcc_list", "hcc_details_summary"]].head(10)

## Step 5: Visualize Score Distribution & Top HCCs

We can plot visualizations to understand the overall profile of our population's risk scores and identify the most common HCC categories.

In [ ]:
import matplotlib.pyplot as plt

# Clean modern layout styling
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Distribution of Risk Scores
axes[0].hist(successful_rows["risk_score"], bins=35, color="#1e88e5", edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution of HCC Risk Scores (RAF)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Risk Adjustment Factor (RAF) Score", fontsize=12)
axes[0].set_ylabel("Number of Patients", fontsize=12)

# 2. Top HCCs
hcc_frequencies = {}
for codes in successful_rows["hcc_list"].dropna():
    for code in str(codes).split(','):
        if code.strip():
            hcc_frequencies[code.strip()] = hcc_frequencies.get(code.strip(), 0) + 1
            
top_hccs_sorted = sorted(hcc_frequencies.items(), key=lambda x: x[1], reverse=True)[:10]
if top_hccs_sorted:
    categories, counts = zip(*top_hccs_sorted)
    axes[1].bar(categories, counts, color="#26a69a", edgecolor="black", alpha=0.7)
    axes[1].set_title("Top 10 Most Common HCC Categories", fontsize=14, fontweight="bold")
    axes[1].set_xlabel("HCC Category", fontsize=12)
    axes[1].set_ylabel("Number of Patients", fontsize=12)
    for idx, val in enumerate(counts):
        axes[1].text(idx, val + (max(counts)*0.01), f"{val:,}", ha='center', fontweight='bold')
else:
    axes[1].text(0.5, 0.5, "No HCCs detected", ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()